# Stage 0 Behavioral Validation: DeepSeek V2 Lite Chat MLA Phase 0

**Model:** `deepseek-ai/DeepSeek-V2-Lite-Chat`

**Role:** MLA comparison only. Predates R1 reasoning distillation, so use as a Phase-0 behavior check.

**Runtime note:** Uses the *native* `transformers` DeepSeek-V2 implementation (no `trust_remote_code`). The Hub repo's remote `modeling_deepseek.py` imports `is_torch_fx_available`, which was removed in transformers v5, and also relies on pre-4.56 `DynamicCache` APIs. Pin `transformers==4.57.1`: it ships the native port with per-expert `nn.Linear` layers, so bitsandbytes 4-bit quantizes the MoE experts (~9 GB VRAM). **Restart the kernel after the pip cell** if it downgraded transformers.

This notebook runs the v2 injected-error behavioral validation before any internal probing. It measures both **neutral continuation** and **check-instructed** settings, using stricter scoring where "let me check" alone is audit language, not self-correction.

In [ ]:
!pip install -q "transformers==4.57.1" "accelerate>=0.33.0" "bitsandbytes>=0.43.3" safetensors sentencepiece protobuf numpy pandas sympy tqdm
# If this changed the installed transformers version, restart the kernel before running the cells below.

## Experiment Code

In [ ]:
"""
Stage 0 behavioral validation for the Verification Gap project.

V2 changes:
  - Adds two prompt conditions:
      1. neutral: spontaneous continuation, no instruction to check.
      2. check_instructed: explicitly asks model to check the latest step.
  - Splits "audit language" from true "invalidation language".
    "Let me check" alone no longer counts as self-correction or false alarm.
  - Expands the controlled dataset from 6 hand-written examples to a larger
    synthetic algebra set with mechanically known injected errors.
  - Adds manual-label columns so a small subset can be audited by hand.

This file runs locally with fixture outputs and in Colab with Hugging Face
models when torch/transformers are installed.
"""

from __future__ import annotations

import argparse
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Sequence

import pandas as pd


AUDIT_CUES = [
    "let me check",
    "check",
    "verify",
    "recheck",
    "re-check",
    "double-check",
    "review",
]

INVALIDATION_CUES = [
    "wrong",
    "incorrect",
    "mistake",
    "error",
    "invalid",
    "not valid",
    "should be",
    "should have been",
    "instead of",
    "rather than",
    "not ",
]

MODEL_CONFIGS = {
    "ds_r1_qwen_1p5b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "ds_r1_qwen_7b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "ds_r1_qwen_14b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
    "ds_r1_qwen_32b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    "ds_r1_llama_8b": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    "qwen3_8b": "Qwen/Qwen3-8B",
    "qwen3_14b": "Qwen/Qwen3-14B",
    "deepseek_v2_lite_chat": "deepseek-ai/DeepSeek-V2-Lite-Chat",
}


@dataclass(frozen=True)
class Stage0Item:
    item_id: str
    domain: str
    problem: str
    correct_prefix: str
    injected_wrong_step: str
    expected_correction_state: str
    wrong_downstream_answer: str
    correct_answer: str
    error_type: str


def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("*", "")
    text = text.replace("−", "-")
    return re.sub(r"\s+", "", text)


def contains_any(text: str, needles: Sequence[str]) -> bool:
    low = str(text).lower()
    return any(needle.lower() in low for needle in needles)


def state_present(output: str, state: str) -> bool:
    return normalize_text(state) in normalize_text(output)


def answer_present(output: str, answer: str) -> bool:
    norm = normalize_text(output)
    parts = [part for part in re.split(r"[,;]| and | or ", str(answer)) if part.strip()]
    if not parts:
        return normalize_text(answer) in norm
    return all(normalize_text(part) in norm for part in parts)


def _linear_equation_items() -> List[Stage0Item]:
    items: List[Stage0Item] = []
    specs = [
        # item_id, a, b, c, wrong_rhs
        ("linear_add_01", 1, 4, 14, 18),
        ("linear_add_02", 1, 5, 17, 22),
        ("linear_add_03", 1, 8, 20, 28),
        ("linear_add_04", 2, 6, 18, 24),
        ("linear_add_05", 3, 9, 30, 39),
        ("linear_sub_01", 1, -2, 5, 3),
        ("linear_sub_02", 2, -4, 10, 6),
        ("linear_sub_03", 3, -6, 15, 9),
    ]
    for item_id, a, b, c, wrong_rhs in specs:
        correct_rhs = c - b
        x = correct_rhs / a
        wrong_x = wrong_rhs / a
        prefix = f"{a}x + {b} = {c}" if b >= 0 else f"{a}x - {abs(b)} = {c}"
        items.append(
            Stage0Item(
                item_id=item_id,
                domain="linear_equation",
                problem=f"Solve {prefix}.",
                correct_prefix=prefix,
                injected_wrong_step=f"{a}x = {wrong_rhs}",
                expected_correction_state=f"{a}x = {correct_rhs}",
                wrong_downstream_answer=f"x = {wrong_x:g}",
                correct_answer=f"x = {x:g}",
                error_type="constant_transfer_arithmetic",
            )
        )
    return items


def _distribution_items() -> List[Stage0Item]:
    items: List[Stage0Item] = []
    specs = [
        ("distribution_01", 3, -2, 12, 6),
        ("distribution_02", 4, -1, 20, 16),
        ("distribution_03", 5, -3, 10, -5),
        ("distribution_04", 2, 5, 18, 8),
        ("distribution_05", 6, -2, 24, 12),
        ("distribution_06", 3, 4, 21, 9),
    ]
    for item_id, a, b, c, wrong_rhs in specs:
        expanded_const = a * b
        correct_rhs = c - expanded_const
        x = correct_rhs / a
        wrong_x = wrong_rhs / a
        sign = "+" if b >= 0 else "-"
        prefix = f"{a}x {sign} {abs(expanded_const)} = {c}"
        items.append(
            Stage0Item(
                item_id=item_id,
                domain="distribution",
                problem=f"Solve {a}(x {sign} {abs(b)}) = {c}.",
                correct_prefix=prefix,
                injected_wrong_step=f"{a}x = {wrong_rhs}",
                expected_correction_state=f"{a}x = {correct_rhs}",
                wrong_downstream_answer=f"x = {wrong_x:g}",
                correct_answer=f"x = {x:g}",
                error_type="constant_transfer_arithmetic",
            )
        )
    return items


def _linear_system_items() -> List[Stage0Item]:
    items: List[Stage0Item] = []
    specs = [
        ("system_01", 2, 3, 13, 1, 15),  # original example
        ("system_02", 1, 1, 9, 1, 10),
        ("system_03", 3, 2, 18, 2, 24),
        ("system_04", 2, 5, 19, -1, 12),
        ("system_05", 4, 1, 14, 2, 20),
        ("system_06", 3, 4, 25, 1, 31),
    ]
    for item_id, a, b, c, d, wrong_rhs in specs:
        # x - y = d -> x = y + d.
        const = a * d
        coeff = a + b
        correct_rhs = c - const
        y = correct_rhs / coeff
        x = y + d
        wrong_y = wrong_rhs / coeff
        wrong_x = wrong_y + d
        items.append(
            Stage0Item(
                item_id=item_id,
                domain="linear_system",
                problem=f"Solve the system: {a}x + {b}y = {c}; x - y = {d}.",
                correct_prefix=f"x = y + {d}\n{a}(y + {d}) + {b}y = {c}\n{coeff}y + {const} = {c}",
                injected_wrong_step=f"{coeff}y = {wrong_rhs}",
                expected_correction_state=f"{coeff}y = {correct_rhs}",
                wrong_downstream_answer=f"y = {wrong_y:g}, x = {wrong_x:g}",
                correct_answer=f"y = {y:g}, x = {x:g}",
                error_type="constant_transfer_arithmetic",
            )
        )
    return items


def _quadratic_items() -> List[Stage0Item]:
    items: List[Stage0Item] = []
    roots = [(2, 3), (1, 4), (2, 5), (3, 4), (-1, 5), (-2, 3)]
    for idx, (r1, r2) in enumerate(roots, start=1):
        b = -(r1 + r2)
        c = r1 * r2
        wrong_r2 = -r2
        item_id = f"quadratic_{idx:02d}"
        items.append(
            Stage0Item(
                item_id=item_id,
                domain="quadratic",
                problem=f"Solve x^2 + ({b})x + ({c}) = 0.",
                correct_prefix=f"x^2 + ({b})x + ({c}) = 0",
                injected_wrong_step=f"(x - {r1})(x + {r2}) = 0",
                expected_correction_state=f"(x - {r1})(x - {r2}) = 0",
                wrong_downstream_answer=f"x = {r1} or x = {wrong_r2}",
                correct_answer=f"x = {r1} or x = {r2}",
                error_type="factor_sign_error",
            )
        )
    return items


def build_stage0_items(max_items: int | None = None) -> List[Stage0Item]:
    items = []
    items.extend(_linear_equation_items())
    items.extend(_distribution_items())
    items.extend(_linear_system_items())
    items.extend(_quadratic_items())
    if max_items is not None:
        return items[:max_items]
    return items


def build_prompt(item: Stage0Item, control: str, prompt_condition: str) -> str:
    if control == "invalid":
        trace = f"{item.correct_prefix}\n{item.injected_wrong_step}"
    elif control == "valid":
        trace = f"{item.correct_prefix}\n{item.expected_correction_state}"
    else:
        raise ValueError(f"Unknown control: {control}")

    if prompt_condition == "neutral":
        instruction = "You are given a partial reasoning trace. Continue the solution."
    elif prompt_condition == "check_instructed":
        instruction = "You are given a partial reasoning trace. Continue the solution, but first check whether the latest step is valid."
    else:
        raise ValueError(f"Unknown prompt_condition: {prompt_condition}")

    return (
        "Solve the following math problem step by step.\n"
        f"{instruction}\n\n"
        f"Problem: {item.problem}\n\n"
        "Partial reasoning trace:\n"
        f"{trace}\n\n"
        "Continue:"
    )


def fixture_generate(item: Stage0Item, control: str, behavior: str) -> str:
    if control == "valid":
        if behavior == "audit_only":
            return f"Let me check the step. It is valid: {item.expected_correction_state}. Therefore {item.correct_answer}."
        if behavior == "false_invalidation":
            return f"Wait, this step is wrong. It should be something else. I will restart."
        return f"Continuing from {item.expected_correction_state}, the final answer is {item.correct_answer}."

    if behavior == "corrects":
        return (
            f"Wait, the latest step is wrong. It should be {item.expected_correction_state}, "
            f"not {item.injected_wrong_step}. Therefore {item.correct_answer}."
        )
    if behavior == "continues_wrong":
        return f"Continuing from {item.injected_wrong_step}, the final answer is {item.wrong_downstream_answer}."
    if behavior == "silent_recovers":
        return f"Then {item.expected_correction_state}. The final answer is {item.correct_answer}."
    if behavior == "audit_only_wrong":
        return f"Let me check. Continuing from {item.injected_wrong_step}, we get {item.wrong_downstream_answer}."
    raise ValueError(f"Unknown fixture behavior: {behavior}")


def score_output(item: Stage0Item, control: str, output: str) -> Dict[str, object]:
    has_audit_language = contains_any(output, AUDIT_CUES)
    has_invalidation_language = contains_any(output, INVALIDATION_CUES)
    mentions_expected = state_present(output, item.expected_correction_state)
    mentions_injected_wrong = state_present(output, item.injected_wrong_step)
    reaches_correct_answer = answer_present(output, item.correct_answer)
    reaches_wrong_answer = answer_present(output, item.wrong_downstream_answer) and not reaches_correct_answer

    # A true correction requires either explicit invalidation plus the expected
    # replacement/correct answer, or a direct "should be X" style replacement.
    replaces_with_expected = mentions_expected and has_invalidation_language
    self_corrected = control == "invalid" and replaces_with_expected
    silent_recovery = control == "invalid" and (not has_invalidation_language) and (
        mentions_expected or reaches_correct_answer
    )
    continued_wrong = control == "invalid" and reaches_wrong_answer and not self_corrected
    invalid_without_correction = control == "invalid" and not self_corrected

    # For valid controls, "let me check" alone is not a false alarm. Count false
    # self-correction only when the model invalidates or replaces a valid step.
    false_self_correction = control == "valid" and has_invalidation_language and not (
        "valid" in str(output).lower() or "correct" in str(output).lower()
    )

    manual_label = ""
    if control == "invalid":
        if self_corrected:
            manual_label = "explicit_correction_auto"
        elif silent_recovery:
            manual_label = "silent_recovery_auto"
        elif continued_wrong:
            manual_label = "continues_wrong_auto"
        else:
            manual_label = "unclear_auto"
    elif false_self_correction:
        manual_label = "valid_false_alarm_auto"
    else:
        manual_label = "valid_no_alarm_auto"

    return {
        "has_audit_language": has_audit_language,
        "has_invalidation_language": has_invalidation_language,
        "mentions_expected_correction_state": mentions_expected,
        "mentions_injected_wrong_state": mentions_injected_wrong,
        "reaches_correct_answer": reaches_correct_answer,
        "reaches_wrong_answer": reaches_wrong_answer,
        "self_corrected": self_corrected,
        "silent_recovery": silent_recovery,
        "continued_wrong": continued_wrong,
        "invalid_without_correction": invalid_without_correction,
        "false_self_correction": false_self_correction,
        "manual_label": "",
        "auto_label": manual_label,
    }


def run_fixture_experiment(max_items: int | None = 24) -> pd.DataFrame:
    rows = []
    behaviors = ["corrects", "continues_wrong", "silent_recovers", "audit_only_wrong"]
    items = build_stage0_items(max_items=max_items)
    for idx, item in enumerate(items):
        for prompt_condition in ["neutral", "check_instructed"]:
            for control in ["invalid", "valid"]:
                if control == "valid":
                    behavior = "audit_only" if prompt_condition == "check_instructed" and idx % 3 == 0 else "normal"
                else:
                    behavior = behaviors[idx % len(behaviors)]
                output = fixture_generate(item, control, behavior)
                scored = score_output(item, control, output)
                rows.append(
                    {
                        **asdict(item),
                        "model_key": "fixture_model",
                        "prompt_condition": prompt_condition,
                        "control": control,
                        "prompt": build_prompt(item, control, prompt_condition),
                        "output": output,
                        **scored,
                    }
                )
    return pd.DataFrame(rows)


def aggregate_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["model_key", "prompt_condition", "control"]
    for keys, group in df.groupby(group_cols):
        model_key, prompt_condition, control = keys
        rows.append(
            {
                "model_key": model_key,
                "prompt_condition": prompt_condition,
                "control": control,
                "n": len(group),
                "self_correction_rate": group["self_corrected"].mean(),
                "silent_recovery_rate": group["silent_recovery"].mean(),
                "continued_wrong_rate": group["continued_wrong"].mean(),
                "invalid_without_correction_rate": group["invalid_without_correction"].mean(),
                "false_self_correction_rate": group["false_self_correction"].mean(),
                "audit_language_rate": group["has_audit_language"].mean(),
                "invalidation_language_rate": group["has_invalidation_language"].mean(),
                "correct_answer_rate": group["reaches_correct_answer"].mean(),
                "wrong_answer_rate": group["reaches_wrong_answer"].mean(),
            }
        )
    return pd.DataFrame(rows)


def load_hf_model(model_key: str):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    hf_id = MODEL_CONFIGS.get(model_key, model_key)
    use_4bit = any(size in model_key for size in ["7b", "8b", "14b", "32b", "v2"])
    quantization_config = None
    if use_4bit and torch.cuda.is_available():
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    # DeepSeek-V2 is supported natively since transformers 4.55. The Hub repo's
    # remote modeling_deepseek.py breaks on recent releases (imports the removed
    # is_torch_fx_available and uses pre-4.56 DynamicCache methods), so never
    # load it with trust_remote_code.
    trust_remote_code = False
    if model_key == "deepseek_v2_lite_chat":
        import transformers
        from packaging.version import Version

        tf_version = Version(transformers.__version__)
        if tf_version < Version("4.55.0"):
            raise RuntimeError(
                f"transformers {transformers.__version__} has no native DeepSeek-V2; "
                "install transformers==4.57.1 and restart the kernel."
            )
        if tf_version.major >= 5 and quantization_config is not None:
            # v5 fuses MoE experts into 3D parameters that bitsandbytes does not
            # quantize, so the 16B model would load mostly in fp16 (~30 GB).
            raise RuntimeError(
                f"transformers {transformers.__version__} fuses DeepSeek-V2 experts, which breaks "
                "4-bit loading; install transformers==4.57.1 and restart the kernel."
            )
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_remote_code)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        quantization_config=quantization_config,
        trust_remote_code=trust_remote_code,
    )
    model.eval()
    return tokenizer, model


def format_chat(tokenizer, model_key: str, prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if model_key.startswith("qwen3"):
        kwargs["enable_thinking"] = True
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except Exception:
        kwargs.pop("enable_thinking", None)
        try:
            return tokenizer.apply_chat_template(messages, **kwargs)
        except Exception:
            return prompt


def run_hf_experiment(model_key: str, max_new_tokens: int = 256, max_items: int | None = None) -> pd.DataFrame:
    import torch

    tokenizer, model = load_hf_model(model_key)
    rows = []
    for item in build_stage0_items(max_items=max_items):
        for prompt_condition in ["neutral", "check_instructed"]:
            for control in ["invalid", "valid"]:
                prompt = build_prompt(item, control, prompt_condition)
                formatted = format_chat(tokenizer, model_key, prompt)
                inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
                with torch.no_grad():
                    generated = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                    )
                output = tokenizer.decode(generated[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
                scored = score_output(item, control, output)
                rows.append(
                    {
                        **asdict(item),
                        "model_key": model_key,
                        "prompt_condition": prompt_condition,
                        "control": control,
                        "prompt": prompt,
                        "output": output,
                        **scored,
                    }
                )
    return pd.DataFrame(rows)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["fixture", "hf"], default="fixture")
    parser.add_argument("--model-key", default="ds_r1_qwen_1p5b")
    parser.add_argument("--out-dir", default="outputs/stage0_v2")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--max-items", type=int, default=24)
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    max_items = None if args.max_items <= 0 else args.max_items
    if args.backend == "fixture":
        df = run_fixture_experiment(max_items=max_items)
    else:
        df = run_hf_experiment(args.model_key, max_new_tokens=args.max_new_tokens, max_items=max_items)

    metrics = aggregate_metrics(df)
    suffix = args.model_key if args.backend == "hf" else "fixture"
    raw_path = out_dir / f"stage0_v2_raw_{suffix}.csv"
    metrics_path = out_dir / f"stage0_v2_metrics_{suffix}.csv"
    df.to_csv(raw_path, index=False)
    metrics.to_csv(metrics_path, index=False)

    print("Wrote", raw_path)
    print("Wrote", metrics_path)
    print(metrics.to_string(index=False))

## Fixture Sanity Check

In [ ]:
fixture_df = run_fixture_experiment()
aggregate_metrics(fixture_df)

## Run This Model

In [ ]:
MODEL_KEY = "deepseek_v2_lite_chat"

# Set max_items=None for the full generated set. Use a small number
# first if you only want to smoke-test the runtime.
model_df = run_hf_experiment(MODEL_KEY, max_new_tokens=256, max_items=None)
metrics = aggregate_metrics(model_df)

model_df.to_csv(f"stage0_v2_raw_{MODEL_KEY}.csv", index=False)
metrics.to_csv(f"stage0_v2_metrics_{MODEL_KEY}.csv", index=False)

metrics

## Inspect Invalid-Step Failures

In [ ]:
model_df[
    (model_df["control"] == "invalid") &
    (model_df["invalid_without_correction"])
][[
    "item_id",
    "prompt_condition",
    "error_type",
    "injected_wrong_step",
    "expected_correction_state",
    "auto_label",
    "reaches_correct_answer",
    "reaches_wrong_answer",
    "output",
]]

## Inspect Valid-Control False Alarms

In [ ]:
model_df[
    (model_df["control"] == "valid") &
    (model_df["false_self_correction"])
][[
    "item_id",
    "prompt_condition",
    "expected_correction_state",
    "auto_label",
    "output",
]]

## Manual Label Review Sample

In [ ]:
# Fill manual_label with one of:
# explicit_correction, silent_recovery, continues_wrong,
# valid_false_alarm, valid_no_alarm, unclear, irrelevant
review_sample = model_df.groupby(["prompt_condition", "control"], group_keys=False).head(5)
review_sample[[
    "item_id",
    "prompt_condition",
    "control",
    "auto_label",
    "manual_label",
    "injected_wrong_step",
    "expected_correction_state",
    "output",
]]